# Trend-following strategy

This code lays the foundation for a trend-following strategy.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import polars as pl

NOTEBOOK_FILE = globals().get("__vsc_ipynb_file__")
WORKSPACE_FOLDER = Path(NOTEBOOK_FILE).resolve().parents[1] if NOTEBOOK_FILE else Path.cwd()
sys.path.insert(0, str(WORKSPACE_FOLDER))

from evaluation import calc_key_figures, plot_key_figures

DATE_COL = "AsOfDate"
PRICES = pl.read_csv(WORKSPACE_FOLDER / "example_prices.csv", try_parse_dates=True).rename({"dates": DATE_COL})
INSTRUMENTS = [c for c in PRICES.columns if c != DATE_COL]

### Strategy summarized
* Calculate the signal for the last M days:

    $$X_{T, j} = \text{sign}\left(\sum\limits_{k = T - M + 1}^T{r_{k, j}}\right) = \text{sign}(P_{T, j} - P_{T - M, j})$$

    Note that the signal for a given asset will be $\pm 1$ (apart from exact ties).

* Estimate the volatility of the assets at time $T$ from the last N days:
    $$\sigma_{T,j} = \sqrt{\frac{1}{N}\sum_{k=T-N+1}^{T} r_{k,j}^2}$$

* Calculate the positions $\tilde X_{T,j}$ that would make each asset contribute approximately equal risk to the portfolio (ignoring correlations between assets):
    $$\tilde X_{T,j} = \frac{X_{T,j}}{\sigma_{T,j}}$$

* The returns from  these positions need to be calculated taking into account the two-day lag:
    $$R_T = \sum_j \tilde X_{T-2,j}\, r_{T,j}$$

* The next step is to calculate the portfolio risk from holding equally-risk-weighted positions:
    $$\hat\sigma_T = \text{std}\left(R_{T-N+1}, \dots, R_T\right)$$

* We can then calculate the final portfolio positions as:
    $$\hat X_{T,j} = \frac{\tilde X_{T,j}}{\hat\sigma_T}$$

    Dividing by $\sigma_{T,j}$ balances risk between assets, and dividing by $\hat\sigma_T$ keeps the risk of the whole strategy steady over time. Defining the risk of an asset as the size of our position multiplied by its volatility, and using $|X_{T,j}| = 1$:

    $$\text{risk}_{T+2,j} = \sigma^{\text{true}}_{T+2,j} \cdot |\hat X_{T,j}| = \sigma^{\text{true}}_{T+2,j} \cdot \frac{|X_{T,j}|}{\sigma_{T,j}\,\hat\sigma_T} \approx \frac{1}{\hat\sigma_T}$$

    Every asset therefore carries about the same risk, $1/\hat\sigma_T$.

* The realized return from the portfolio will be:

    $$\hat R_{T+2} = \sum_j \hat X_{T,j}\, r_{T+2,j} = \frac{1}{\hat\sigma_T}\sum_j \tilde X_{T,j}\, r_{T+2,j} = \frac{R_{T+2}}{\hat\sigma_T}$$

    Let $\sigma^{\text{true}}_{T+2}$ be the true, unobserved volatility of $R_{T+2}$, given the data through day $T$. Since $\hat\sigma_T$ is known on day $T$, it acts as a constant:

    $$\text{Var}\left(\hat R_{T+2} \mid \text{data through } T\right) = \frac{\text{Var}\left(R_{T+2} \mid \text{data through } T\right)}{\hat\sigma_T^2} = \frac{\left(\sigma^{\text{true}}_{T+2}\right)^2}{\hat\sigma_T^2} \approx 1$$

    The final return therefore has a standard deviation of about 1, provided $\hat\sigma_T$ is a good forecast of $\sigma^{\text{true}}_{T+2}$.

* The trend-following comes from holding the direction of the last $M$ days. We go long if the price has risen and short if it has fallen, so the return on asset $j$ is:

    $$\hat X_{T,j}\, r_{T+2,j} = \frac{\text{sign}(P_{T,j} - P_{T-M,j})\, r_{T+2,j}}{\sigma_{T,j}\,\hat\sigma_T}$$

    This is positive when the return on day $T+2$ has the same sign as the past trend, and negative when it goes against it. The strategy therefore makes money on average if:

    $$E\left[\text{sign}(P_{T,j} - P_{T-M,j})\, r_{T+2,j}\right] > 0$$

    This requires price trends to persist. The autocorrelation analysis in `modeling_example.ipynb` supports this, with positive autocorrelation for blocks of about 20 days or more.

